In [70]:
import pandas as pd
df = pd.read_csv('output.csv')

In [62]:
df

,Thai,Viet
0,- รร.แกรนด์โฮเต็ล,- Khách sạn Grand.
1,- สายไม่ว่าง,- Đường dây bận.
2,นั่นใครพูดคะ,Ai đấy?
3,ฉันจะโอนสายคุณ ไปที่แผนกรูมเซอร์วิส,Tôi có thể kết nối quý khách với dịch vụ phòng.
4,ฮัลโหลๆ,A lô.
...,...,...
17001633,นั่นใครน่ะ,Ai thế?
17001634,มิราห์,Mirah?
17001635,เธอตามอาเบลกลับมาบ้านเหรอ,Cô đi theo Abel về nhà phải không?
17001636,เธอต้องการอะไร,Cô muốn gì?


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.6.0+cpu
False


In [63]:
# Check for missing data
df.isnull().sum()

Thai    0
Viet    0
dtype: int64

In [64]:
# length sequence
df['thai_len'] = df['Thai'].str.split().str.len()
df['viet_len'] = df['Viet'].str.split().str.len()
df[['thai_len', 'viet_len']].describe()


,thai_len,viet_len
count,1.700164e+07,1.700164e+07
mean,1.864862e+00,7.783794e+00
std,1.069208e+00,5.317196e+00
min,1.000000e+00,1.000000e+00
25%,1.000000e+00,4.000000e+00
50%,2.000000e+00,7.000000e+00
75%,2.000000e+00,1.000000e+01
max,4.300000e+01,3.180000e+02


In [75]:
df.duplicated().sum()                         


np.int64(0)

In [76]:
df.duplicated(subset='Viet').sum()      

np.int64(1860982)

In [77]:
df.duplicated(subset='Thai').sum()      


np.int64(1877126)

In [74]:
df = df.drop_duplicates()

In [78]:
df

,Thai,Viet
0,- รร.แกรนด์โฮเต็ล,- Khách sạn Grand.
1,- สายไม่ว่าง,- Đường dây bận.
2,นั่นใครพูดคะ,Ai đấy?
3,ฉันจะโอนสายคุณ ไปที่แผนกรูมเซอร์วิส,Tôi có thể kết nối quý khách với dịch vụ phòng.
4,ฮัลโหลๆ,A lô.
...,...,...
17001623,ทำไมพระเจ้าถึงมอบความสามารถนี้ให้กับเรา,Tại sao Chúa lại ban cho bọn cháu khả năng này?
17001625,ไม่ว่าพระเจ้าจะให้อะไรเรามา จะดีหรือร้าย เราก็...,"Dù Chúa ban gì cho ta, dù tốt hay xấu, thì ta ..."
17001627,เพราะมัน เราจึงได้มีชีวิตอย่างไร้ซึ่งสิ่งติดค้าง,"Vì có nó, ta có thể sống một cuộc sống không c..."
17001628,พวกเราที่ได้เปิดดวงตาที่สามนั้น มีชีวิตอยู่ก็เ...,Những người đã mở con mắt thứ ba như chúng ta ...


In [3]:
import pandas as pd
df = pd.read_csv('/home/leloc/Document/USTH/Thesis/Data/preprocess.csv')
df

,Thai,Viet
0,- รร.แกรนด์โฮเต็ล,- Khách sạn Grand.
1,- สายไม่ว่าง,- Đường dây bận.
2,นั่นใครพูดคะ,Ai đấy?
3,ฉันจะโอนสายคุณ ไปที่แผนกรูมเซอร์วิส,Tôi có thể kết nối quý khách với dịch vụ phòng.
4,ฮัลโหลๆ,A lô.
...,...,...
15193987,ทำไมพระเจ้าถึงมอบความสามารถนี้ให้กับเรา,Tại sao Chúa lại ban cho bọn cháu khả năng này?
15193988,ไม่ว่าพระเจ้าจะให้อะไรเรามา จะดีหรือร้าย เราก็...,"Dù Chúa ban gì cho ta, dù tốt hay xấu, thì ta ..."
15193989,เพราะมัน เราจึงได้มีชีวิตอย่างไร้ซึ่งสิ่งติดค้าง,"Vì có nó, ta có thể sống một cuộc sống không c..."
15193990,พวกเราที่ได้เปิดดวงตาที่สามนั้น มีชีวิตอยู่ก็เ...,Những người đã mở con mắt thứ ba như chúng ta ...


In [ ]:
# Advanced preprocessing optimized for Thai-Vietnamese
import pandas as pd
import re
from tqdm import tqdm
import numpy as np
import gc  # for garbage collection
import os  # to check file exists

df = pd.read_csv('/home/leloc/Document/USTH/Thesis/Data/preprocess.csv')

def remove_subtitle_tags(text):
    """Loại bỏ các tag phụ đề phổ biến trong OpenSubtitles"""
    # Các tag thường gặp trong phụ đề
    subtitle_tags = [
        r'\[.*?\]',  # [âm thanh], [nhạc], ...
        r'\(.*?\)',  # (tiếng cười), (thở dài), ...
        r'<.*?>',    # <i>italic text</i>
        r'\{.*?\}',   # {hiệu ứng}
        r'##\s*\w+',  # ## nhân vật
        r'♪.*?♪',     # ♪ nhạc nền ♪
        r'^\W+',      # Ký tự không phải chữ ở đầu câu
        r'\W+$',      # Ký tự không phải chữ ở cuối câu
    ]
    
    for pattern in subtitle_tags:
        text = re.sub(pattern, ' ', text)
    return text.strip()

def advanced_clean_text(text, is_thai=False):
    try:
        text = str(text)
    except Exception:
        return None

    if pd.isna(text) or len(text.strip()) == 0:
        return None

    # Bước 1: Kiểm tra độ dài TRƯỚC để tối ưu hiệu suất
    word_count = len(text.split())
    if word_count < 5 or word_count > 150:
        return None

    # Bước 2: Loại bỏ tag phụ đề
    text = remove_subtitle_tags(text)
    if len(text.strip()) == 0:
        return None

    # Bước 3: Kiểm tra ngôn ngữ
    if is_thai:
        if not re.search(r'[\u0E00-\u0E7F]', text):
            return None
    else:
        if not re.search(r'[a-zA-ZÀ-ỹ]', text):
            return None

    # Bước 4: Thêm khoảng trắng sau ký tự đặc biệt (chỉ cho tiếng Việt)
    if not is_thai:
        text = re.sub(r'([A-Za-zÀ-ỹ])([^\sA-Za-zÀ-ỹ])', r'\1 \2', text)

    # Bước 5: Loại bỏ ký tự đặc biệt
    if is_thai:
        text = re.sub(r'[^\w\s\u0E00-\u0E7F]', ' ', text)
    else:
        text = re.sub(r'[^\w\s\u00C0-\u1EF9]', ' ', text)

    # Bước 6: Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()

    # Kiểm tra lại độ dài cuối cùng (phòng trường hợp thay đổi sau xử lý)
    final_word_count = len(text.split())
    if 5 <= final_word_count <= 150:
        return text
    else:
        return None

    
    
def process_chunk(df_chunk):
    tqdm.pandas(desc="Cleaning Thai")
    df_chunk['Thai_cleaned'] = df_chunk['Thai'].progress_apply(lambda x: advanced_clean_text(x, is_thai=True))

    tqdm.pandas(desc="Cleaning Vietnamese")
    df_chunk['Viet_cleaned'] = df_chunk['Viet'].progress_apply(advanced_clean_text)

    # Loại bỏ ngay các dòng có None hoặc chuỗi rỗng sau xử lý
    valid_mask = (
        df_chunk['Thai_cleaned'].notnull() & df_chunk['Viet_cleaned'].notnull() &
        (df_chunk['Thai_cleaned'].str.len() > 0) & (df_chunk['Viet_cleaned'].str.len() > 0)
    )

    removed = df_chunk[~valid_mask][['Thai', 'Viet']]
    cleaned = df_chunk[valid_mask][['Thai_cleaned', 'Viet_cleaned']].rename(
        columns={'Thai_cleaned': 'Thai', 'Viet_cleaned': 'Viet'}
    )

    return cleaned, removed


# File paths
input_path = '/home/leloc/Document/USTH/Thesis/Data/preprocess.csv'
output_cleaned = '/home/leloc/Document/USTH/Thesis/Data/preprocessed_cleaned_chunks.csv'
output_removed = '/home/leloc/Document/USTH/Thesis/Data/removed_rows_chunks.csv'

# Xóa file cũ nếu có để tránh ghi đè sai định dạng
for path in [output_cleaned, output_removed]:
    if os.path.exists(path):
        os.remove(path)

# Chunk processing only
chunk_size = 10000
total_cleaned = 0
total_removed = 0
first_chunk = True

chunks = pd.read_csv(input_path, chunksize=chunk_size, usecols=['Thai', 'Viet'])

for chunk in tqdm(chunks, desc="Processing chunks"):
    # Ensure strings only for current chunk
    chunk['Thai'] = chunk['Thai'].astype(str)
    chunk['Viet'] = chunk['Viet'].astype(str)

    cleaned_chunk, removed_chunk = process_chunk(chunk)
    
    # Append to output files
    cleaned_chunk.to_csv(output_cleaned, index=False, mode='a', header=first_chunk)
    removed_chunk.to_csv(output_removed, index=False, mode='a', header=first_chunk)
    
    total_cleaned += len(cleaned_chunk)
    total_removed += len(removed_chunk)
    first_chunk = False

    # Free memory
    del chunk, cleaned_chunk, removed_chunk
    gc.collect()

print(f"✅ Chunk processing completed.")
print(f"👉 Total cleaned rows: {total_cleaned}")
print(f"👉 Total removed rows: {total_removed}")


Cleaning Vietnamese: 100%|██████████| 10000/10000 [00:00<00:00, 90477.36it/s]
Processing chunks: 587it [02:44,  3.57it/s]


KeyboardInterrupt: 

In [4]:
df_1 = pd.read_csv('/home/leloc/Document/USTH/Thesis/Data/removed_rows_chunks.csv')
len(df_1)

14854878

In [5]:
df_1

,Thai,Viet
0,- รร.แกรนด์โฮเต็ล,- Khách sạn Grand.
1,- สายไม่ว่าง,- Đường dây bận.
2,นั่นใครพูดคะ,Ai đấy?
3,ฉันจะโอนสายคุณ ไปที่แผนกรูมเซอร์วิส,Tôi có thể kết nối quý khách với dịch vụ phòng.
4,ฮัลโหลๆ,A lô.
...,...,...
14854873,ทำไมพระเจ้าถึงมอบความสามารถนี้ให้กับเรา,Tại sao Chúa lại ban cho bọn cháu khả năng này?
14854874,ไม่ว่าพระเจ้าจะให้อะไรเรามา จะดีหรือร้าย เราก็...,"Dù Chúa ban gì cho ta, dù tốt hay xấu, thì ta ..."
14854875,เพราะมัน เราจึงได้มีชีวิตอย่างไร้ซึ่งสิ่งติดค้าง,"Vì có nó, ta có thể sống một cuộc sống không c..."
14854876,พวกเราที่ได้เปิดดวงตาที่สามนั้น มีชีวิตอยู่ก็เ...,Những người đã mở con mắt thứ ba như chúng ta ...


In [3]:
import pandas as pd
df_split = pd.read_csv('/home/leloc/Document/USTH/Thesis/Data/preprocessed_with_split.csv')
df_split.head

<bound method NDFrame.head of                                                      Thai  \
0       ฉันรู้ว่าคุณเคยบอกว่า จะคอยส่งข่าวฉันเรื่อยๆ แ...   
1                เคียว ฉันก็มี มีด ฉันก็มี ดูของฉันซะก่อน   
2              แดเนียล เลอรอย แม็คแคบบี้ ที่ 3 สามีฉันเอง   
3                           แพร์รี่ มันยุค 90 นะ ใครสนล่ะ   
4       เธ เธฑเธ เธ เน เธฐ เธ เน เธฒเธ เน เธญเธขเธฒเธ ...   
...                                                   ...   
339109  ผู้พันมาร์ควิสเผาทหาร 47 คนทั้งเป็น เพื่อให้นิ...   
339110  บูลส์จบฤดูกาลด้วยสถิติชนะ 72 ครั้ง แพ้สิบครั้ง...   
339111  สตาลินกราดถูกสร้างขึ้นบนตลิ่ง ข้างแม่น้ำวอลก้า...   
339112           9834 And then I say bo add a B 9834 9834   
339113  ดูเหมือนว่าวิสัยทัศน์ของท่านส ส ชา จะเคยเข้ากั...   

                                                     Viet  split  
0       Tôi biết bà đã nói sẽ báo cho tôi nhưng tuần n...  train  
1       Đủ loại đặc thù nhé Có lưỡi hái tử thần Lưỡi h...  train  
2       Chắc cậu có nghe nói đến anh

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Ngôn ngữ nguồn và đích
tokenizer.src_lang = "tha_Thai"
target_lang_token = "vie_Latn"
forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_lang_token)

# Câu tiếng Thái cần dịch
text = "สวัสดีครับ ยินดีที่ได้รู้จัก"

# Token hóa đầu vào
inputs = tokenizer(text, return_tensors="pt")

# Dịch sang tiếng Việt
translated_tokens = model.generate(
    **inputs, forced_bos_token_id=forced_bos_token_id
)

# Giải mã kết quả
print(tokenizer.decode(translated_tokens[0], skip_special_tokens=True))


/home/leloc/anaconda3/envs/translate/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chào, rất vui được gặp anh.


In [5]:
import re

# Thêm cột "split"
train_df['split'] = 'train'
dev_df['split'] = 'dev'
test_df['split'] = 'test'

# Gộp lại thành một DataFrame
full_df = pd.concat([train_df, dev_df, test_df])

def word_tokenize(text):
    return re.findall(r'(\w+|[^\w\s])', text)

# Lọc dữ liệu 'train' từ full_df
train_df = full_df[full_df['split'] == 'train'].copy()

# Lấy mẫu 10.000 dòng
smpl = train_df.sample(10000, random_state=1).copy()

# Tokenize
smpl['thai_toks'] = smpl['Thai'].apply(tokenizer.tokenize)
smpl['viet_toks'] = smpl['Viet'].apply(tokenizer.tokenize)

# Word tokenize bằng regex
smpl['thai_words'] = smpl['Thai'].apply(word_tokenize)
smpl['viet_words'] = smpl['Viet'].apply(word_tokenize)


In [6]:
stats = smpl[
    ['thai_toks', 'viet_toks', 'thai_words', 'viet_words']
].applymap(len).describe()
print(stats.thai_toks['mean'] / stats.thai_words['mean'])  # 2.0349
print(stats.viet_toks['mean'] / stats.viet_words['mean'])  # 2.4234
stats

0.8188747023500583
1.0988421012632117


/tmp/ipykernel_17942/3997695083.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  ].applymap(len).describe()


,thai_toks,viet_toks,thai_words,viet_words
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,26.032600,20.242100,31.790700,18.421300
std,11.322092,10.387997,14.246311,9.724128
min,5.000000,5.000000,5.000000,5.000000
25%,18.000000,13.000000,21.000000,11.000000
50%,24.000000,18.000000,30.000000,16.500000
75%,33.000000,26.000000,41.000000,24.000000
max,119.000000,123.000000,96.000000,95.000000


In [7]:
import random
from tqdm.auto import tqdm

# Giả sử bạn muốn check cột tiếng Thái, thay trans_df.tyv thành trans_df.Thai nếu cần
texts_with_unk = [
    text for text in tqdm(df_split.Thai)  # hoặc trans_df.Thai nếu muốn check tiếng Thái
    if tokenizer.unk_token_id in tokenizer(text).input_ids
]

print(f"Số câu chứa token <unk>: {len(texts_with_unk)}")

# Lấy 5 câu ngẫu nhiên có token <unk> để xem
sample_texts = random.sample(texts_with_unk, min(5, len(texts_with_unk)))
print(sample_texts)


100%|██████████| 339114/339114 [00:42<00:00, 7977.19it/s] 

Số câu chứa token <unk>: 1476
['ณป ภฯภบ ภฺณืธฆ ภโพฦผญ ภ ย๗ด๋ทฮ ม๘วเวฯฑโธธ วฯธ้ ตศดูณื ฑืฐิ ดูพ฿', 'กระหม่อมฌอง เดอ คารูจส์ อัศวิน เป็นผู้อุทธรณ์ในศาลของฝ่าบาท', 'พวกเขาสละชีพเพื่อปกป้องบุรุษ ᵽvk qã slà ciᵽ ᵽew opkp og búrúş สตรีและเด็ก stri lế dé k ที่จะไม่มีวันรู้ชื่อพวกเขา ţi jà mâ mivánru cw oᵽvk qã', 'ณป ธปภบ ลฐฟ๎ดูธ้ วา ภฯภฬ ธนพฦม๚ฒจถ๕ดู ธนภบ รฅภำฐจภฬ ตฺต๛ธฃฐํ ดฯ ป ศฐภฬณช ด๋วะป ศฐฟกตต ฟตวโภป ณขฤฅฒจพ฿', 'ความลับของท่าน xvamláb qog ţ an ปลอดภัยกับพวกเรา ท่านหญฺิง plodƀáykábᵽvk rã ţ anhñฺ íg']


In [19]:
import re
import sys
import unicodedata
from sacremoses import MosesPunctNormalizer
import regex as re

mpn = MosesPunctNormalizer(lang="en")

# Recompile patterns with standard re
mpn.substitutions = [
    (re.compile(r.pattern) if hasattr(r, 'pattern') else r, sub)
    for r, sub in mpn.substitutions
]

def get_non_printing_char_replacer(replace_by: str = " "):
    non_printable_map = {
        ord(c): replace_by
        for c in (chr(i) for i in range(sys.maxunicode + 1))
        if unicodedata.category(c) in {"C", "Cc", "Cf", "Cs", "Co", "Cn"}
    }

    def replace_non_printing_char(line) -> str:
        return line.translate(non_printable_map)

    return replace_non_printing_char

replace_nonprint = get_non_printing_char_replacer(" ")

def remove_strange_unicode(text):
    # Loại bỏ ký tự ngoài Basic Multilingual Plane (BMP)
    text = re.sub(r'[^\u0000-\uFFFF]', '', text)
    # Loại bỏ ký tự Unicode mở rộng không phải chữ cái, số, dấu câu hoặc khoảng trắng
    text = re.sub(r'[^\p{Thai}\p{Z}]', '', text)  # Chỉ giữ lại ký tự Thái và khoảng trắng
    return text

def is_thai_char(c):
    return '\u0E00' <= c <= '\u0E7F'

def is_all_thai(text):
    # Loại bỏ khoảng trắng và dấu câu tiếng Thái trước khi kiểm tra
    text_no_space = re.sub(r'[\s\u0E00-\u0E7F\p{P}]', '', text, flags=re.UNICODE)
    if len(text_no_space) > 0:
        return False
    # Kiểm tra xem còn ký tự nào không phải tiếng Thái không
    return not re.search(r'[^\u0E00-\u0E7F\s]', text)

def preproc(text):
    if not isinstance(text, str):
        return ""
    clean = mpn.normalize(text)
    clean = replace_nonprint(clean)
    clean = unicodedata.normalize("NFKC", clean)
    clean = remove_strange_unicode(clean)
    return clean.strip()

def clean_thai_only(text):
    text = preproc(text)
    if not text or not is_all_thai(text):
        return None
    return text

# Áp dụng lên dataframe df_split với cột 'Thai'
df_split['cleaned_text'] = df_split['Thai'].apply(clean_thai_only)
df_split = df_split[df_split['cleaned_text'].notnull()].reset_index(drop=True)

print(f"Số dòng sau khi lọc và tiền xử lý: {len(df_split)}")

Số dòng sau khi lọc và tiền xử lý: 207866


In [20]:
texts_with_unk_normed = [
    text for text in tqdm(texts_with_unk) 
    if tokenizer.unk_token_id in tokenizer(preproc(text)).input_ids
]
print(len(texts_with_unk_normed))

100%|██████████| 1476/1476 [00:00<00:00, 4220.85it/s]

1076


In [21]:
for i, text in enumerate(texts_with_unk[:10]):
    pre = preproc(text)
    ids = tokenizer(pre).input_ids
    print(f"\n--- Sample {i} ---")
    print("Before:", text)
    print("After:", pre)
    print("IDs:", ids)
    print("UNKs:", [i for i in ids if i == tokenizer.unk_token_id])



--- Sample 0 ---
Before: ถ้ ถ้ ๘ ข์ ขาใหตวามเคารพสกั้อตเเลนต
After: ถ้ ถ้ ๘ ข์ ขาใหตวามเคารพสกั้อตเเลนต
IDs: [256175, 18319, 248577, 18319, 248577, 248059, 3, 18898, 249540, 18898, 248399, 6014, 248992, 248736, 5095, 19988, 4168, 249098, 249039, 248611, 10535, 248523, 248992, 248552, 10052, 248439, 248992, 2]
UNKs: [3]

--- Sample 1 ---
Before: ตอนที่ฌอน มิลเลอร์หนี มีคนเห็นมันครั้งสุดท้าย ที่โซดิแอก ไปที่ช่องแคบตอนลมแรง ซึ่งไม่เข้าเค้า ผมเลยเช็คเรือสินค้า ที่ออกจากท่าวั้นนั้น
After: ตอนที่ฌอน มิลเลอร์หนี มีคนเห็นมันครั้งสุดท้าย ที่โซดิแอก ไปที่ช่องแคบตอนลมแรง ซึ่งไม่เข้าเค้า ผมเลยเช็คเรือสินค้า ที่ออกจากท่าวั้นนั้น
IDs: [256175, 53362, 2679, 3, 11012, 41667, 66381, 10052, 26558, 159947, 25060, 9043, 30471, 7313, 45177, 176964, 11824, 216408, 160038, 248972, 7753, 48847, 2679, 12440, 2099, 248972, 248836, 248988, 45343, 248792, 248647, 121205, 52123, 4408, 29229, 106011, 14658, 20702, 26543, 196943, 195465, 95606, 237852, 11824, 145463, 248842, 41173, 26157, 9384, 2]
UNKs: [3]

--- Sa